# 온도 데이터의 시작·끝 시각과 기록이 끊긴 구간 찾기

**담당: 김태구**  ·  관련 Issue: #10 (번호를 채우세요)

## 이 노트북에서 할 일

온도 CSV가 **언제부터 언제까지** 기록되었는지 확인하고,
중간에 **기록이 뚝 끊긴 구간**이 있는지 찾아냅니다.

## 왜 하는지

기록이 끊긴 구간을 모르고 분석하면, 그냥 데이터가 없는 것을 "온도가 갑자기 변했다"고 잘못 읽을 수 있습니다.
또 나중에 학습 기간과 검증 기간을 나눌 때, 끊긴 구간을 피해서 잘라야 합니다.

## 진행 방법

1. 아래 칸을 **위에서부터 순서대로** 실행하세요.
2. `# TODO` 라고 적힌 곳을 직접 채우세요. 막히면 디스코드에 물어보세요.
3. 맨 아래 **결과 정리** 칸에 확인한 값을 한국어 문장으로 적으세요. 이게 진짜 결과물입니다.
4. 다 했으면 커밋 전에 맨 마지막 안내를 읽으세요.

> 데이터 파일이 없으면 `data/` 폴더에 CSV 3개를 먼저 넣으세요. 이 파일들은 Git에 올라가지 않습니다(공유받은 자료를 각자 직접 넣습니다).

## 공통 준비

In [2]:
# 이 칸은 그대로 실행하세요. 데이터 경로를 잡아둡니다.
import os
import pandas as pd

DATA_DIR = os.path.join("..", "data")          # notebooks 폴더 기준 한 단계 위의 data 폴더
TEMP_CSV   = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV  = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")

pd.set_option("display.max_columns", 50)
print("경로 확인:", os.path.exists(TEMP_CSV), os.path.exists(EVENT_CSV), os.path.exists(REPAIR_CSV))

경로 확인: True True True


### 1단계 — 데이터를 읽고 시각 컬럼을 날짜 형식으로 바꾸세요

In [24]:
df = pd.read_csv(TEMP_CSV, encoding="utf-8")

# print(df.head()) # 정렬 전

# TODO: MEAS_DT를 날짜/시간 형식으로 바꾸고, 시간순으로 정렬하세요.
# 힌트: pd.to_datetime(...) 그리고 .sort_values("MEAS_DT")
df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"])
df = df.sort_values("MEAS_DT")
print(df.head()) # 정렬 후

print(df.shape)

              MEAS_DT  LOT_NO   CLN-IN    TC-Z1    TC-Z2    OP-Z1    OP-Z2  \
0 2024-01-02 03:36:00  240001  69.0111  100.559  99.7062  66.6165  18.5975   
1 2024-01-02 03:36:01  240001  69.0724  100.497  99.6447  73.2873  20.9099   
2 2024-01-02 03:36:02  240001  68.9497  100.497  99.7062  70.7377  20.9651   
3 2024-01-02 03:36:03  240001  68.8884  100.497  99.7062  70.4484  18.7313   
4 2024-01-02 03:36:04  240001  68.8884  100.436  99.7062  70.0716  21.1505   

     TC-Z3    TC-Z4    TC-Z5    TC-Z6    OP-Z3    OP-Z4    OP-Z5    OP-Z6  \
0  859.683  860.800  860.367  860.126  83.9075  47.2134  51.6966  69.0189   
1  859.621  860.737  860.429  860.126  88.8233  48.6681  50.1850  69.0560   
2  859.683  860.737  860.367  860.126  84.2600  48.5541  51.7643  69.0900   
3  859.683  860.675  860.367  860.188  84.5850  49.9122  51.7661  67.6582   
4  859.683  860.737  860.367  860.126  84.8836  48.2923  51.7676  70.7738   

      CP-Z4        CP-Z4M    TC-Z7    TC-Z8    TC-Z9    OP-Z7    OP-

### 2단계 — 첫 시각과 마지막 시각을 확인하세요

`.min()` 과 `.max()` 를 쓰면 됩니다. 두 값을 빼면 전체 기간 길이가 나옵니다.

In [18]:
# TODO: 첫 시각, 마지막 시각, 그 차이를 출력하세요.
# 힌트: df["MEAS_DT"].min(), df["MEAS_DT"].max()
df_min = df["MEAS_DT"].min()
df_max = df["MEAS_DT"].max()
print("첫 시각:", df_min)
print("마지막 시각:", df_max)
print("차이:", df_max - df_min)

첫 시각: 2024-01-02 03:36:00
마지막 시각: 2024-12-28 16:44:24
차이: 361 days 13:08:24


### 3단계 — 기록이 끊긴 구간을 찾으세요

정상 간격(정수진 담당자가 확인한 값, 보통 1초)보다 **훨씬 크게 벌어진 곳**이 끊긴 구간입니다.

여기서는 "정상 간격의 10배가 넘으면 끊긴 것으로 본다" 정도로 기준을 잡고 시작해 보세요.
기준은 데이터를 보면서 조정해도 됩니다. **어떤 기준을 썼는지 아래 결과 정리에 꼭 적으세요.**

In [32]:
gap = df["MEAS_DT"].diff()
gap_sec = gap.dt.total_seconds()

NORMAL_SEC = 1        # TODO: 실제 확인한 정상 간격으로 바꾸세요
THRESHOLD = NORMAL_SEC * 10   # TODO: 필요하면 기준을 조정하세요

# TODO: gap_sec 이 THRESHOLD 보다 큰 행만 골라내세요.
# 힌트: big = df[gap_sec > THRESHOLD]

print(gap_sec.value_counts().head())
big = df[gap_sec > THRESHOLD]
print("끊긴 구간:", len(big), "개")

MEAS_DT
1.0     334966
2.0      10372
3.0         31
11.0         3
6.0          2
Name: count, dtype: int64
끊긴 구간: 16 개


### 4단계 — 끊긴 구간을 "언제부터 언제까지"로 정리하세요

3단계에서 찾은 행은 **끊김이 끝난 시점**입니다.
끊김이 시작된 시점은 그 **바로 앞 행**의 시각입니다.

In [31]:
# TODO: 끊긴 구간을 표로 만들어 보세요.
# 힌트:
# idx = df.index[gap_sec > THRESHOLD]
# for i in idx:
#     start = df.loc[i - 1, "MEAS_DT"]
#     end   = df.loc[i, "MEAS_DT"]
#     print(start, "->", end, "  길이:", end - start)

idx = df.index[gap_sec > THRESHOLD]
for i in idx:
    start = df.loc[i - 1, "MEAS_DT"]
    end   = df.loc[i, "MEAS_DT"]
    print("시작된 시점:",start, "->", "끝난 시점:",end, "  길이:", end - start)

시작된 시점: 2024-01-02 15:04:30 -> 끝난 시점: 2024-03-07 21:22:00   길이: 65 days 06:17:30
시작된 시점: 2024-03-08 04:44:55 -> 끝난 시점: 2024-03-08 04:45:06   길이: 0 days 00:00:11
시작된 시점: 2024-03-08 06:29:48 -> 끝난 시점: 2024-04-13 23:16:00   길이: 36 days 16:46:12
시작된 시점: 2024-04-14 09:27:19 -> 끝난 시점: 2024-04-16 11:35:00   길이: 2 days 02:07:41
시작된 시점: 2024-04-16 13:39:56 -> 끝난 시점: 2024-05-03 18:50:00   길이: 17 days 05:10:04
시작된 시점: 2024-05-03 20:04:38 -> 끝난 시점: 2024-06-09 05:26:00   길이: 36 days 09:21:22
시작된 시점: 2024-06-09 09:02:53 -> 끝난 시점: 2024-06-15 01:04:00   길이: 5 days 16:01:07
시작된 시점: 2024-06-15 04:49:23 -> 끝난 시점: 2024-07-07 16:57:00   길이: 22 days 12:07:37
시작된 시점: 2024-07-07 18:41:45 -> 끝난 시점: 2024-07-07 18:41:56   길이: 0 days 00:00:11
시작된 시점: 2024-07-08 03:18:46 -> 끝난 시점: 2024-07-08 03:18:57   길이: 0 days 00:00:11
시작된 시점: 2024-07-08 06:29:34 -> 끝난 시점: 2024-09-08 22:03:00   길이: 62 days 15:33:26
시작된 시점: 2024-09-09 08:44:12 -> 끝난 시점: 2024-10-18 04:05:00   길이: 38 days 19:20:48
시작된 시점: 2024-10-18 11:50:53 -> 끝난

## 결과 정리 — 여기를 꼭 채우세요

아래 빈칸을 확인한 값으로 바꿔서 적으세요. 이 내용이 `docs/02-data-contract.md`로 옮겨집니다.

| 항목 | 확인한 값 |
| --- | --- |
| 첫 기록 시각 | 2024-01-02 03:36:00 |
| 마지막 기록 시각 | 2024-12-28 16:44:24 |
| 전체 기간 길이 | 약 361 일 |
| 기록이 끊긴 구간 개수 | 16 개 |
| 가장 긴 끊김 | ____ 부터 ____ 까지, 약 ____ 시간 |


### 이상하다고 느낀 점 / 확실하지 않은 점

- (있으면 적어주세요. 없으면 "없음"이라고 적으세요.)

---

## 커밋하기 전에 읽으세요

1. **출력은 지우지 않아도 됩니다.** `nbstripout` 필터를 등록해 뒀다면 `git add` 할 때 자동으로 지워집니다.
   등록했는지 확인: `python -m nbstripout --status` → `Automatic cleanup enabled` 가 나와야 합니다.
   안 나오면: `python -m nbstripout --install --attributes .gitattributes`
2. 이 노트북 **한 파일만** 커밋하세요. 다른 사람 파일은 건드리지 마세요.
3. 브랜치를 만들어서 작업하세요. 예) `git switch -c feat/이슈번호-설명`
4. 커밋 메시지 첫 줄: `feat: 온도 데이터 기간과 결측 구간 확인`
5. PR 본문에 `Closes #이슈번호` 를 꼭 적으세요.